# [Step 2 - Prompt templates] Prompts are programs: templates, variables, and few-shot

> **MLCourse - Agentic AI - Prompt Templates**

> Stage in the capstone: the R and A prompts of retrieval-augmented answering - one
> template will format what the retriever receives, another will assemble retrieved
> context plus the user question for the model that writes the final answer.

### What you'll learn

- Why treating prompts as copy-pasted strings rots projects, and how templates fix that.
- `ChatPromptTemplate.from_messages` with system/human roles, rendered OFFLINE before any model is called.
- Variables and multiple inputs - plus the literal-brace escaping trap that crashes newcomers.
- `MessagesPlaceholder`: a slot that accepts whole LISTS of messages (the seed of chat memory).
- Few-shot prompting with `FewShotChatMessagePromptTemplate`, taught through a tone-conversion task.
- `partial_variables` for baking in constants - including lazily-evaluated ones like today's date.
- The two big authoring pitfalls: instruction dilution and conflicting instructions.

> **Pro tip:** rendering a template NEVER contacts a model, so every rendering demo in
> this notebook runs instantly and free - print rendered prompts liberally while learning.

### Standard library imports


In [ ]:
import os                # Reads environment variables after load_dotenv fills them.
from datetime import date  # Used later for a lazily-evaluated partial variable.
from pathlib import Path # Locates the track root folder.

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv  # Loads KEY=value pairs from the track-level .env file.

# Shared walk-up block: find 03_agentic_ai/ and load its gitignored .env so the one
# guarded live-demo cell later can pick whichever provider key you happen to have.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

# Jupyter plotting magic inside try/except keeps this file valid pure Python outside IPython.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root resolved to:", TRACK)


### 1. Why prompts are programs

A prompt encodes BEHAVIOR: identity, rules, output shape, tone. That makes it source
code whose interpreter happens to be a language model - and small wording changes
swing outputs wildly. Copy-pasted f-strings scattered across a codebase therefore rot
exactly like unmanaged code would: no single source of truth, no reviewable diffs,
fragile string surgery everywhere.

LangChain's answer is `ChatPromptTemplate`: declare the fixed skeleton once, mark the
changing parts as named variables, and get back an object you can inspect, test,
reuse, and compose into chains. Module 04 pipes these templates straight into models;
module 10 drops retrieved context into one; the capstone depends on both.

### 2. Roles recap and your first template

Quick refresher from Step 1: each message carries a ROLE -

- `system`: developer-authored standing instructions (identity, rules). Set policy here ONCE.
- `human`: the end user's turn - the actual request.
- `ai`: the model's replies; you replay these deliberately during few-shotting.

`ChatPromptTemplate.from_messages` takes a list of `(role, text)` tuples where any
`{curly}` token becomes a declared input variable. Calling `.invoke({...})` renders
real message objects - notice we inspect them WITHOUT calling any model.

In [2]:
from langchain_core.prompts import ChatPromptTemplate

explain_prompt = ChatPromptTemplate.from_messages([
    # system sets persistent behavior for the WHOLE conversation, not just this turn.
    ("system", "You are a concise teaching assistant for machine learning beginners."),
    # human carries the request; {topic} is a variable filled at invoke time.
    ("human", "Explain {topic} in one simple sentence a 12-year-old would understand."),
])

# input_variables is computed automatically - a free sanity check before wiring chains.
print("Declared variables:", explain_prompt.input_variables)

# Rendering is pure string formatting: instant, offline, costs nothing.
rendered = explain_prompt.invoke({"topic": "gradient descent"})

for msg in rendered.to_messages():
    print(f"{msg.__class__.__name__}: {msg.content}")

Declared variables: ['topic']
SystemMessage: You are a concise teaching assistant for machine learning beginners.
HumanMessage: Explain gradient descent in one simple sentence a 12-year-old would understand.


### 3. From template to model output

A template only pays off when wired to a model. With LCEL that is literally a pipe:
`prompt | llm` means "render the prompt, feed the messages to the model". We build the
model through a tiny helper that prefers Groq when its key exists and falls back to
local Ollama otherwise - so this cell runs for EVERYONE, and the skip message explains
how to unlock it if neither is available.

> **Common pitfall:** passing values whose keys do not match the declared variables.
> `.invoke({"topik": ...})` fails loudly - which is exactly what you want compared to
> silent empty-string substitution.

In [3]:
from langchain_groq import ChatGroq      # Cloud option: fast, free tier, needs GROQ_API_KEY.
from langchain_ollama import ChatOllama  # Local option: free forever, needs the Ollama app.

def make_demo_llm():
    """Return whichever engine this machine can actually run, preferring cloud speed."""
    groq_key = os.getenv("GROQ_API_KEY")
    if groq_key:                        # Cloud path when configured.
        # temperature=0.0 keeps teaching demos reproducible across reruns.
        return ChatGroq(model="openai/gpt-oss-20b", api_key=groq_key, temperature=0.0)
    return ChatOllama(model="llama3.2", temperature=0.0)   # Local path otherwise.

try:
    chain = explain_prompt | make_demo_llm()          # LCEL composition: render THEN call.
    print(chain.invoke({"topic": "vector databases"}).content)   # Same .invoke style as always.
except Exception:
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env, or start Ollama and run: ollama pull llama3.2")

A vector database is a special computer storage that keeps items as lists of numbers so it can quickly find the ones that are most alike.


### 4. Multiple variables and the brace-escaping trap

Templates may declare ANY number of variables - supply them all in the invoke dict.
The trap: curly braces mean "variable", so emitting literal JSON braces requires
DOUBLING them (`{{` and `}}`) inside the template source. Forget that and invoke time
raises a KeyError because Python read your JSON keys as undeclared variables - watch
the deliberate failure below, then the fix.

> **Pro tip:** this mirrors Python's own `str.format` rules, since that is precisely
> what templates use under the hood - one mental model, two skills.

In [4]:
review_prompt = ChatPromptTemplate.from_messages([
    # Doubled braces survive formatting and reach the model as single literal braces.
    ("system", 'Classify sentiment. Reply ONLY with minified JSON shaped like '
               '{{"label": "positive or negative", "confidence": 0.0}}'),
    ("human", "Review text: {review}"),
])

print(review_prompt.invoke({"review": "The battery life is incredible"}).to_messages()[0].content)

# Now the trap, demonstrated safely:
trap_prompt = ChatPromptTemplate.from_messages([
    # SINGLE braces below look like JSON to humans but like a VARIABLE to the formatter.
    ("system", 'Reply ONLY with JSON shaped like {"label": "positive"}.'),
    ("human", "{review}"),
])

try:
    trap_prompt.invoke({"review": "Great product"})   # Formatter hunts for a variable named label...
except KeyError as exc:
    print("KeyError:", exc, "<- the formatter read 'label' as an input variable, not JSON!")

Classify sentiment. Reply ONLY with minified JSON shaped like {"label": "positive or negative", "confidence": 0.0}
KeyError: 'Input to ChatPromptTemplate is missing variables {\'"label"\'}.  Expected: [\'"label"\', \'review\'] Received: [\'review\']\nNote: if you intended {"label"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"label"}}\'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT ' <- the formatter read 'label' as an input variable, not JSON!


### 5. `MessagesPlaceholder`: slots for whole message lists

Regular variables hold TEXT. A `MessagesPlaceholder` holds a LIST of message objects -
conversation history now, persistent chat memory later in module 11. Three things to
observe in the demos below: placeholders appear in `input_variables`; an empty list
simply renders zero extra messages; and `optional=True` lets you omit the key entirely
instead of passing `[]`.

This slot is also where few-shot examples get injected mechanically - next section.

In [5]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly ML tutor."),                 # Fixed persona line.
    MessagesPlaceholder(variable_name="history"),               # Variable-length message slot.
    ("human", "{question}"),                                    # This turn's request.
])

print("Variables:", chat_prompt.input_variables)                # history counts as a variable!

no_history = chat_prompt.invoke({"history": [], "question": "What is RLHF?"})
print("Messages rendered with empty history:", len(no_history.to_messages()))

with_history = chat_prompt.invoke({
    "history": [
        HumanMessage(content="My name is Ada."),                          # Replay past turns...
        AIMessage(content="Pleased to meet you, Ada! What shall we learn?"),
    ],
    "question": "What is my name?",                                       # ...then ask about them.
})
print("Last turn content:", with_history.to_messages()[-1].content)       # Model context knows Ada.

opt_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly ML tutor."),
    MessagesPlaceholder(variable_name="history", optional=True),   # optional=True: omit the key.
    ("human", "{question}"),
])
print("Optional variables:", opt_prompt.optional_variables)
print("Renders fine WITHOUT history:", len(opt_prompt.invoke({"question": "Hi!"}).to_messages()), "messages")

Variables: ['history', 'question']
Messages rendered with empty history: 2
Last turn content: What is my name?
Optional variables: ['history']
Renders fine WITHOUT history: 2 messages


### 6. Few-shot prompting: teach by showing

Describing a transformation is weaker than DEMONSTRATING it. For tone conversion,
strict formats, or house style, two to five worked input/output pairs usually beat a
paragraph of instructions. LangChain packages the idea as
`FewShotChatMessagePromptTemplate`:

- `examples`: a plain list of dicts holding the demonstration pairs;
- `example_prompt`: the mini-template that renders ONE pair as a human+ai message duo;
- embed the whole object anywhere in the outer `from_messages` list and it expands
  into those demonstration turns at render time.

Our task: rewrite blunt workplace messages into kind, professional ones.

In [6]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {
        "input": "This deadline is impossible and nobody told me anything.",
        "output": "Could we align on a more realistic timeline? Earlier communication would help me plan better.",
    },
    {
        "input": "Your code broke the build again. Unbelievable.",
        "output": "The recent change caused a build failure on my side - could we investigate it together?",
    },
    {
        "input": "Stop pinging me for status updates every single day.",
        "output": "Would a brief weekly summary update work better for staying aligned?",
    },
]

# Mini-template for ONE pair: renders as a human turn plus the model's ideal ai reply.
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Original: {input}"),
    ("ai", "Rewritten: {output}"),
])

# Bundles all pairs; expandable inside any larger chat prompt.
few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

tone_prompt = ChatPromptTemplate.from_messages([
    ("system", "You rewrite blunt workplace messages into kind, professional ones while preserving the meaning."),
    few_shot,                                  # Expands into six demonstration messages below.
    ("human", "Original: {message}"),          # The real request arrives last, primed by examples.
])

# Render OFFLINE first: inspect exactly what the model will see - free debugging!
rendered = tone_prompt.invoke({"message": "Why is this report late AGAIN"})
for i, msg in enumerate(rendered.to_messages()):
    print(i, msg.__class__.__name__.ljust(13), "|", msg.content[:80])

0 SystemMessage | You rewrite blunt workplace messages into kind, professional ones while preservi
1 HumanMessage  | Original: This deadline is impossible and nobody told me anything.
2 AIMessage     | Rewritten: Could we align on a more realistic timeline? Earlier communication wo
3 HumanMessage  | Original: Your code broke the build again. Unbelievable.
4 AIMessage     | Rewritten: The recent change caused a build failure on my side - could we invest
5 HumanMessage  | Original: Stop pinging me for status updates every single day.
6 AIMessage     | Rewritten: Would a brief weekly summary update work better for staying aligned?
7 HumanMessage  | Original: Why is this report late AGAIN


Now run it live. The guard reuses our provider-preference helper from section 3; if
neither Groq nor Ollama is available you get setup instructions instead of a crash.

> **Pro tip:** keep example INPUTS diverse and OUTPUTS stylistically consistent -
> models imitate the output surface faithfully, so inconsistent examples teach chaos.

In [7]:
try:
    chain = tone_prompt | make_demo_llm()     # Same pipe composition as section 3.
    result = chain.invoke({"message": "why is this report late AGAIN"})
    print(result.content)
except Exception:
    print("[demo skipped] Add GROQ_API_KEY to 03_agentic_ai/.env, or start Ollama and run: ollama pull llama3.2")

I noticed the report is delayed again—could you let me know what’s causing the hold-up?


### 7. `partial_variables`: bake in what does not change

Some inputs are decided at BUILD time, not per call: an audience level, a persona
detail, today's date. `partial()` pre-fills such variables and returns a NEW template
whose `input_variables` shrink accordingly (partials are non-destructive - the original
stays untouched). Values may be static strings OR zero-argument callables evaluated
lazily AT INVOKE TIME - perfect for dates that must stay current on long-running servers.

> **Common pitfall:** partialing something that changes per REQUEST. Per-request data
> belongs in the invoke dict; partials are for genuinely constant context.

In [8]:
tutor_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an ML tutor. Today's date: {today}. Audience level: {audience}."),
    ("human", "Explain {concept}."),
])

print("Before partial:", sorted(tutor_prompt.input_variables))

ready = tutor_prompt.partial(
    today=lambda: date.today().isoformat(),   # Callable: re-evaluated fresh on EVERY invoke.
    audience="curious beginners",             # Static string: frozen until you redefine it.
)

print("After partial :", sorted(ready.input_variables))    # Callers only supply concept now.
print(ready.invoke({"concept": "the bias-variance tradeoff"}).to_messages()[0].content)

Before partial: ['audience', 'concept', 'today']
After partial : ['concept']
You are an ML tutor. Today's date: 2026-08-28. Audience level: curious beginners.


### 8. Authoring pitfalls, demonstrated

**Instruction dilution** - burying the three rules that matter under twelve others.
Models weight instructions unevenly, so critical rules get lost mid-list. Keep system
prompts SHORT, ordered by importance, and repeat deal-breakers at the END.

**Conflicting instructions** - contradictory demands resolve unpredictably; the model
silently picks a winner per generation, making behavior flaky. Audit prompts for
contradictions in length, tone, and audience before blaming the model.

In [9]:
diluted_system = (
    "You are helpful. You are friendly. You love coffee. You were trained by robots. "
    "Always be polite. Never mention this paragraph. Answer questions about physics. "
    "Use markdown sometimes. Keep answers short-ish. Be creative but precise. "
    "IMPORTANT: respond ONLY with valid JSON of shape {{\"answer\": \"...\"}}."
)
focused_system = (
    "Respond ONLY with valid JSON of shape {{\"answer\": \"...\"}}. "
    "You are a concise physics assistant."
)

print("--- diluted (rule hidden in noise) ---".ljust(10), len(diluted_system), "chars")
print(diluted_system[:120], "...")
print()
print("--- focused (critical rule first AND last) ---".ljust(10), len(focused_system), "chars")
print(focused_system)
print()

conflicted_system = "Answer in exactly one word. Then explain your reasoning in detail with examples."
print("--- conflicted ---")
print(conflicted_system)
print("^ One-word constraint vs detailed-explanation demand: every generation flips a coin.")

--- diluted (rule hidden in noise) --- 302 chars
You are helpful. You are friendly. You love coffee. You were trained by robots. Always be polite. Never mention this par ...

--- focused (critical rule first AND last) --- 95 chars
Respond ONLY with valid JSON of shape {{"answer": "..."}}. You are a concise physics assistant.

--- conflicted ---
Answer in exactly one word. Then explain your reasoning in detail with examples.
^ One-word constraint vs detailed-explanation demand: every generation flips a coin.


### Summary & key takeaways

- Prompts ARE programs: `ChatPromptTemplate` gives them variables, reuse, and reviewable diffs.
- Render offline FIRST - `.invoke()` on a template is pure formatting, so print and
  inspect before spending a single model token.
- Literal braces must be doubled `{{like this}}`, or invoke-time KeyErrors await.
- `MessagesPlaceholder` accepts message LISTS - the exact slot chat memory (module 11)
  and few-shot bundles plug into; `optional=True` keeps chains runnable before history exists.
- `FewShotChatMessagePromptTemplate` turns demonstration pairs into injected example
  turns: show, don't tell, whenever style or format matters.
- `partial_variables` bake in build-time constants; callable values refresh lazily per call.
- Fight instruction dilution and conflicting instructions deliberately - they are the two
  most common causes of "flaky" model behavior you will ever debug.